# English → Kannada Translation using Mistral-7B + QLoRA

## End-to-End Fine-Tuning Project

This notebook demonstrates a complete **Supervised Fine-Tuning (SFT)** pipeline for English → Kannada machine translation using:

- **Mistral-7B-Instruct-v0.3**
- **SFT (Supervised Fine-Tuning)**
- **QLoRA**
- **PEFT / LoRA adapters**
- **4-bit NF4 quantization**
- **AI4Bharat Samanantar dataset**
- **BLEU and chrF evaluation**
- **Base model vs Fine-tuned model comparison**

> ⚠️ Recommended hardware: NVIDIA GPU with sufficient VRAM. The original project was tested with an NVIDIA RTX 3500 Ada Generation Laptop GPU.

---

## Pipeline

```text
English–Kannada Parallel Dataset
            ↓
Dataset Cleaning & Formatting
            ↓
Mistral Instruction / Chat Template
            ↓
4-bit Quantized Mistral Base Model
            ↓
LoRA Adapters
            ↓
SFT + QLoRA Training
            ↓
Save Adapter
            ↓
Inference
            ↓
BLEU + chrF Evaluation
            ↓
Base vs Fine-Tuned Comparison
```


## 1. Install Dependencies

In [1]:
# Uncomment if packages are not installed.
# !pip install -U torch transformers datasets peft trl bitsandbytes accelerate sacrebleu sentencepiece

# For Windows environments, ensure PyTorch is installed with CUDA support
# according to your CUDA / GPU configuration.


## 2. Imports and Configuration

In [2]:
import os
import json
import random
from dataclasses import dataclass
from pathlib import Path

import torch
from datasets import load_dataset, Dataset
from peft import (
    LoraConfig,
    PeftModel,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    set_seed,
)
from trl import SFTTrainer

from sacrebleu.metrics import BLEU, CHRF


c:\Users\hxp2kor\Desktop\Ai_ML\fine_tuning_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1109 21:14:24.189000 34992 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


ModuleNotFoundError: No module named 'sacrebleu'

In [ ]:
@dataclass
class Config:
    # Model
    MODEL_NAME: str = "mistralai/Mistral-7B-Instruct-v0.3"

    # Dataset
    DATASET_NAME: str = "ai4bharat/samanantar"
    DATASET_CONFIG: str = "kn"

    # Sample sizes
    TOTAL_SAMPLES: int = 12000
    TRAIN_SAMPLES: int = 10000
    EVAL_SAMPLES: int = 1000
    TEST_SAMPLES: int = 1000

    # Output paths
    PROJECT_ROOT: Path = Path.cwd()
    OUTPUT_DIR: Path = PROJECT_ROOT / "outputs" / "mistral-en-kn-qlora"
    DATA_DIR: Path = PROJECT_ROOT / "data"

    # QLoRA
    LOAD_IN_4BIT: bool = True
    BNB_4BIT_QUANT_TYPE: str = "nf4"
    BNB_4BIT_USE_DOUBLE_QUANT: bool = True

    # LoRA
    LORA_R: int = 16
    LORA_ALPHA: int = 32
    LORA_DROPOUT: float = 0.05

    TARGET_MODULES: tuple = (
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    )

    # Training
    NUM_TRAIN_EPOCHS: int = 3
    PER_DEVICE_TRAIN_BATCH_SIZE: int = 1
    PER_DEVICE_EVAL_BATCH_SIZE: int = 1
    GRADIENT_ACCUMULATION_STEPS: int = 8
    LEARNING_RATE: float = 2e-4
    MAX_SEQ_LENGTH: int = 512
    WARMUP_STEPS: int = 100
    LOGGING_STEPS: int = 20
    SAVE_STEPS: int = 500

    SEED: int = 42


config = Config()

config.DATA_DIR.mkdir(parents=True, exist_ok=True)
config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Output directory:", config.OUTPUT_DIR)


## 3. Load English–Kannada Parallel Dataset

The AI4Bharat Samanantar Kannada configuration contains columns:

- `src` → English source sentence
- `tgt` → Kannada target sentence


In [ ]:
dataset = load_dataset(
    config.DATASET_NAME,
    config.DATASET_CONFIG,
    split="train",
)

print("Columns:", dataset.column_names)
print("Total available samples:", len(dataset))
print("\nFirst example:")
print(dataset[0])


## 4. Select, Clean and Split the Dataset

In [ ]:
set_seed(config.SEED)

# Select a reproducible subset for this project.
dataset = dataset.shuffle(seed=config.SEED)

dataset = dataset.select(
    range(min(config.TOTAL_SAMPLES, len(dataset)))
)

def format_translation_pair(example):
    return {
        "english": example["src"].strip(),
        "kannada": example["tgt"].strip(),
    }

dataset = dataset.map(
    format_translation_pair,
    remove_columns=dataset.column_names,
)

def is_valid(example):
    english = example["english"]
    kannada = example["kannada"]

    return (
        english is not None
        and kannada is not None
        and len(english.strip()) > 0
        and len(kannada.strip()) > 0
    )

dataset = dataset.filter(is_valid)

print("Valid samples:", len(dataset))
print("\nExample:")
print(dataset[0])


In [ ]:
# Train = 20,000
# Eval  = 2,000
# Test  = 1,000

required = (
    config.TRAIN_SAMPLES
    + config.EVAL_SAMPLES
    + config.TEST_SAMPLES
)

if len(dataset) < required:
    raise ValueError(
        f"Dataset has only {len(dataset)} valid samples, "
        f"but {required} are required."
    )

train_dataset = dataset.select(
    range(0, config.TRAIN_SAMPLES)
)

eval_dataset = dataset.select(
    range(
        config.TRAIN_SAMPLES,
        config.TRAIN_SAMPLES + config.EVAL_SAMPLES,
    )
)

test_dataset = dataset.select(
    range(
        config.TRAIN_SAMPLES + config.EVAL_SAMPLES,
        required,
    )
)

print("Train:", len(train_dataset))
print("Eval:", len(eval_dataset))
print("Test:", len(test_dataset))


## 5. Load Mistral Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    config.MODEL_NAME,
    use_fast=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Tokenizer loaded successfully.")
print("Pad token:", tokenizer.pad_token)


## 6. Format Data for Supervised Fine-Tuning

Each translation pair is converted into the Mistral instruction format:

```text
[INST]
Translate the following English text to Kannada.
Return only the Kannada translation.

English: ...
[/INST]
Kannada translation
```


In [ ]:
def format_for_sft(dataset, tokenizer):

    def format_example(example):
        messages = [
            {
                "role": "user",
                "content": (
                    "Translate the following English text to Kannada. "
                    "Return only the Kannada translation.\n\n"
                    f"English: {example['english']}"
                ),
            },
            {
                "role": "assistant",
                "content": example["kannada"],
            },
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )

        return {"text": text}

    return dataset.map(
        format_example,
        remove_columns=dataset.column_names,
    )


train_sft = format_for_sft(train_dataset, tokenizer)
eval_sft = format_for_sft(eval_dataset, tokenizer)

print(train_sft[0]["text"])


## 7. Configure 4-bit Quantization (QLoRA)

In [ ]:
compute_dtype = torch.bfloat16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=config.LOAD_IN_4BIT,
    bnb_4bit_quant_type=config.BNB_4BIT_QUANT_TYPE,
    bnb_4bit_use_double_quant=config.BNB_4BIT_USE_DOUBLE_QUANT,
    bnb_4bit_compute_dtype=compute_dtype,
)

print(bnb_config)


## 8. Load the Base Mistral Model

In [ ]:
set_seed(config.SEED)

model = AutoModelForCausalLM.from_pretrained(
    config.MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=compute_dtype,
    device_map="auto",
)

# Required for stable k-bit training with gradient checkpointing.
model.config.use_cache = False

model = prepare_model_for_kbit_training(model)

print("Base model loaded successfully.")


## 9. Add LoRA Adapters

In [ ]:
peft_config = LoraConfig(
    r=config.LORA_R,
    lora_alpha=config.LORA_ALPHA,
    lora_dropout=config.LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=list(config.TARGET_MODULES),
)

model = get_peft_model(
    model,
    peft_config,
)

model.print_trainable_parameters()


## 10. Configure SFT Training

Important concepts:

- **Base model:** quantized to 4-bit and frozen
- **LoRA adapters:** trainable
- **QLoRA:** combines quantization + LoRA
- **SFT:** learns from input → expected output examples


In [ ]:
training_args = TrainingArguments(
    output_dir=str(config.OUTPUT_DIR),

    num_train_epochs=config.NUM_TRAIN_EPOCHS,

    per_device_train_batch_size=config.PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=config.PER_DEVICE_EVAL_BATCH_SIZE,

    gradient_accumulation_steps=config.GRADIENT_ACCUMULATION_STEPS,

    learning_rate=config.LEARNING_RATE,

    warmup_steps=config.WARMUP_STEPS,

    logging_steps=config.LOGGING_STEPS,

    save_steps=config.SAVE_STEPS,

    save_total_limit=2,

    fp16=False,
    bf16=True,

    report_to="none",

    remove_unused_columns=False,

    optim="paged_adamw_8bit",

    gradient_checkpointing=True,

    seed=config.SEED,
)

training_args


## 11. Create SFTTrainer

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_sft,
    eval_dataset=eval_sft,
    processing_class=tokenizer,
)

print("SFTTrainer created successfully.")


## 12. Train the Model

> ⏱️ Training time depends on GPU, sequence length, dataset size and epochs.

For a quick demonstration, you can temporarily reduce:
- `TOTAL_SAMPLES`
- `NUM_TRAIN_EPOCHS`


In [ ]:
print("=" * 60)
print("STARTING ENGLISH → KANNADA QLORA TRAINING")
print("=" * 60)

trainer.train()


## 13. Save the LoRA Adapter

In [ ]:
trainer.save_model(str(config.OUTPUT_DIR))
tokenizer.save_pretrained(str(config.OUTPUT_DIR))

print("Saved LoRA adapter to:")
print(config.OUTPUT_DIR)


## 14. Load Fine-Tuned Model for Inference

For inference:

1. Load the quantized base Mistral model.
2. Load the trained LoRA adapter on top of it.
3. Generate the Kannada translation.


In [ ]:
# Optional: free training objects before reloading.
if torch.cuda.is_available():
    torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(
    config.MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=compute_dtype,
    device_map="auto",
)

fine_tuned_model = PeftModel.from_pretrained(
    base_model,
    str(config.OUTPUT_DIR),
)

fine_tuned_model.eval()

print("Fine-tuned model ready.")


## 15. Translation Function

In [ ]:
def build_translation_prompt(english_text):
    messages = [
        {
            "role": "user",
            "content": (
                "Translate the following English text to Kannada. "
                "Return only the Kannada translation.\n\n"
                f"English: {english_text}"
            ),
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


def translate(model, english_text, max_new_tokens=128):
    prompt = build_translation_prompt(english_text)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    translation = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()

    return translation


## 16. Test Manual Translations

In [ ]:
examples = [
    "Hello, how are you?",
    "My name is Pavan.",
    "Artificial intelligence is changing the world.",
    "I am learning machine learning.",
]

for english_text in examples:
    kannada_translation = translate(
        fine_tuned_model,
        english_text,
    )

    print("=" * 60)
    print("English :", english_text)
    print("Kannada :", kannada_translation)


## 17. Load a Separate Base Model for Comparison

We compare:

- **Base Mistral** → before task-specific fine-tuning
- **Fine-tuned Mistral + LoRA** → after English → Kannada SFT

The test set remains untouched during training.


In [ ]:
# Reload a clean base model for comparison.
comparison_base_model = AutoModelForCausalLM.from_pretrained(
    config.MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=compute_dtype,
    device_map="auto",
)

comparison_base_model.eval()

print("Base model ready for comparison.")


## 18. Compare Base vs Fine-Tuned Predictions

In [ ]:
comparison_samples = test_dataset.select(
    range(min(5, len(test_dataset)))
)

for example in comparison_samples:
    english = example["english"]
    reference = example["kannada"]

    base_prediction = translate(
        comparison_base_model,
        english,
    )

    fine_tuned_prediction = translate(
        fine_tuned_model,
        english,
    )

    print("\n" + "=" * 80)
    print("ENGLISH:")
    print(english)

    print("\nREFERENCE:")
    print(reference)

    print("\nBASE MODEL:")
    print(base_prediction)

    print("\nFINE-TUNED MODEL:")
    print(fine_tuned_prediction)


## 19. BLEU and chrF Evaluation

### BLEU
Measures overlap between generated output and reference translations.

### chrF
Character n-gram based metric. It is especially useful for languages where word forms and morphology can vary.

> Higher scores generally indicate closer similarity to the reference translations.


In [ ]:
bleu_metric = BLEU()
chrf_metric = CHRF()

def evaluate_model(model, dataset, max_samples=100):
    references = []
    predictions = []

    evaluation_data = dataset.select(
        range(min(max_samples, len(dataset)))
    )

    for index, example in enumerate(evaluation_data, start=1):
        english = example["english"]
        reference = example["kannada"]

        prediction = translate(
            model,
            english,
        )

        predictions.append(prediction)
        references.append(reference)

        if index % 10 == 0:
            print(f"Evaluated {index}/{len(evaluation_data)}")

    bleu_score = bleu_metric.corpus_score(
        predictions,
        [references],
    )

    chrf_score = chrf_metric.corpus_score(
        predictions,
        [references],
    )

    return {
        "predictions": predictions,
        "references": references,
        "bleu": bleu_score.score,
        "chrf": chrf_score.score,
    }


## 20. Evaluate the Fine-Tuned Model

In [ ]:
fine_tuned_results = evaluate_model(
    fine_tuned_model,
    test_dataset,
    max_samples=100,
)

print("\nFine-Tuned Model Results")
print("=" * 40)
print(f"BLEU: {fine_tuned_results['bleu']:.2f}")
print(f"chrF: {fine_tuned_results['chrf']:.2f}")


## 21. Evaluate the Base Model

In [ ]:
base_results = evaluate_model(
    comparison_base_model,
    test_dataset,
    max_samples=100,
)

print("\nBase Model Results")
print("=" * 40)
print(f"BLEU: {base_results['bleu']:.2f}")
print(f"chrF: {base_results['chrf']:.2f}")


## 22. Final Base vs Fine-Tuned Comparison

In [ ]:
print("=" * 60)
print("BASE MODEL vs FINE-TUNED MODEL")
print("=" * 60)

print("\nBLEU")
print(f"Base Model      : {base_results['bleu']:.2f}")
print(f"Fine-Tuned Model: {fine_tuned_results['bleu']:.2f}")

print("\nchrF")
print(f"Base Model      : {base_results['chrf']:.2f}")
print(f"Fine-Tuned Model: {fine_tuned_results['chrf']:.2f}")


## 23. Save Evaluation Results

In [ ]:
results = {
    "model": config.MODEL_NAME,
    "dataset": config.DATASET_NAME,
    "language_pair": "English → Kannada",
    "training_samples": len(train_dataset),
    "evaluation_samples": len(eval_dataset),
    "test_samples": len(test_dataset),
    "base_model": {
        "bleu": base_results["bleu"],
        "chrf": base_results["chrf"],
    },
    "fine_tuned_model": {
        "bleu": fine_tuned_results["bleu"],
        "chrf": fine_tuned_results["chrf"],
    },
}

results_path = config.OUTPUT_DIR / "evaluation_results.json"

with open(results_path, "w", encoding="utf-8") as file:
    json.dump(
        results,
        file,
        ensure_ascii=False,
        indent=4,
    )

print("Results saved to:")
print(results_path)


# Interview Talking Points

## Problem
General-purpose LLMs may not consistently follow a specific translation format or perform optimally for a specialized language pair.

## Solution
Fine-tune **Mistral-7B-Instruct** for **English → Kannada translation** using **Supervised Fine-Tuning with QLoRA**.

## Why QLoRA?
- Reduces GPU memory requirements through 4-bit quantization.
- Keeps the large base model mostly frozen.
- Trains only lightweight LoRA adapter parameters.
- Makes fine-tuning a 7B model feasible on a single capable GPU.

## Training Pipeline
1. Loaded English–Kannada parallel data.
2. Cleaned and formatted translation pairs.
3. Used Mistral instruction/chat formatting.
4. Loaded Mistral in 4-bit NF4.
5. Added LoRA adapters.
6. Fine-tuned using SFT.
7. Evaluated on an untouched test set.
8. Compared the base and fine-tuned models using BLEU and chrF.

## Key Interview Point
**The fine-tuned model is not trained from scratch.**  
The pretrained Mistral model already contains general language knowledge. Fine-tuning adapts its behavior to the specific English → Kannada translation task.
